In [1]:
from pathlib import Path

import numpy as np
import scipy as sp
import safep
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Suppress future warnings from pandas.
import warnings  
warnings.simplefilter(action="ignore", category=FutureWarning)

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

********* JAX NOT FOUND *********
 PyMBAR can run faster with JAX  
 But will work fine without it   
Either install with pip or conda:
      pip install pybar[jax]     
               OR                
      conda install pymbar       
*********************************


# What this Notebook Does:

Each section of this notebook will calculate a component of the free energy of binding of phenol to lysozyme as described in the companion tutorial. Briefly, during a free energy perturbation simulation, NAMD (or other software) will write the difference in internal energy between the simulated state and an adjacent state (dE =  E_lambda_k+/-1 - E_lambda_k, where lambda_k determines the ensemble being simulated). The functions used by this notebook read, parse, and process those outputs into a standard format that can be quickly analyzed using one of several methods. By default, we use the Bennett Acceptance Ratio (BAR) estimator with automated decorrelation to make the calculations more robust to both outliers and autocorrelation. See Shirts and Chodera (2018) for more details.

One section of the notebook uses [thermodynamic integration (TI)](https://en.wikipedia.org/wiki/Thermodynamic_integration) to calculate the free energy cost of imposing the DBC restraint. This calculation is much more straightforward than FEP calculations. We can analytically determine the derivative of the force with respect to lambda over a series of simulations at discrete values of lambda. Averaging and accumulating those derivatives yields the free energy cost.

# How to Use this Notebook:

## User Parameters:
The notebook as-is will read and process the sample outputs provided. 
To use it for your own data, be sure to update the section labeled "User Settings" below. Pay special attention to the *root* and *path* variables.

- root should be the path (relative or absolute) to the parent directory that contains (or will containt) all your data.
- temperature is the temperature **at which your simulations were run** 
- decorrelate is a flag for automatic decorrelation of samples (see Shirts and Chodera '08). Should be set to True for general use. 
- detectEQ (automatic equilibrium detection). Set to True. This is more robust than manually guessing at the time required for equilibration prior to a FEP run.

## Layout:
The notebook is organized into five sections (separated by horizontal lines):
- Process the Bound Data [(step B)](#bound_fep)
- Process the DBC TI calculation [(step C)](#DBC_TI)
- Process the Unbound Data [(step D)](#unbound_fep)
- Calculate the Volumetric Restraint Contribution [(step E.2)](#volume)
- Calculate the Binding Free Energy [(step E.3)](#total)

## File Structure Data:

```
Repository/Supp-Files
|
|----stepB_alchemy_site
|    |----[sample_]output
|         | *.fepout
|
|----stepC_restraint_perturbation
|    |----[sample_]output
|         | *.colvars.traj
|
|----stepD_alchemy_bulk
|    |----[sample_]output
|         | *fepout
|
|
```



# Other Important Notes and Credits
- This notebook is specially written for the SAFEP tutorial. 
For more up-to-date and general versions see the SAFEP github.

- This and other SAFEP notebooks make use of pyMBAR and Alchemlyb. 
For more information see Shirts and Chodera (2008), ["Statistically optimal analysis of samples from multiple equilibrium states"](https://doi.org/10.1063%2F1.2978177)



# User Settings:

In [2]:
root = "."  # Root path to your project

# Used throughout
temperature = 300
gas_constant = sp.constants.R / (1000 * sp.constants.calorie)
RT = gas_constant * temperature

In [3]:
# Radius of the spherical restraint used for the DBC TI calculation
UW_REGEX = r"(upperWalls[\ \t]+)(\d+.\d+)"
COMfname = (f"{root}/stepC_restraint_perturbation/sample_outputs/DBC_restraint_RFEP.colvars")
COMradius = float(safep.get_num_regex(UW_REGEX, COMfname))

# Position of the DBC upper wall
DBCfname = f"{root}/stepC_restraint_perturbation/inputs/run.namd"
DBCwidth = float(safep.get_num_regex(UW_REGEX, DBCfname))

## Update these paths to point to your output files:

In [4]:
bound_fep_path = Path(f"{root}/stepB_alchemy_site/sample_outputs/")
restraint_perturbation_path = Path(f"{root}/stepC_restraint_perturbation/sample_outputs/")
bulk_fep_path = Path(f"{root}/stepD_alchemy_bulk/sample_outputs/")

## Advanced settings:

In [5]:
detectEQ = True  # Flag for automatic equilibrium detection and decorrelation

***
<a id='bound_fep'></a>
# Process the Bound FEP Data 
Here we process the FEP data generated by decoupling the ligand from the protein (**Step B**)


In [8]:
filepattern = (
    "*.fepout"  # This can be a regex pattern if you have more than one fepout file
)
# Caution: if you have multiple fepout files, name them alphanumerically

# u_nk stores the fep data
u_nk = safep.processing.read_and_process(bound_fep_path.glob(filepattern), #from cell 5
                                         temperature, #from cell 3
                                         False, #do not use alchemlyb's decorrelation function
                                         detectEQ #from cell 6
                                         )

# Run the BAR estimator on the fep data
window_data, cumulative_data = safep.estimators.do_estimation(u_nk)

# Used in the convergence plots
forward, forward_error, backward, backward_error = safep.processing.do_convergence(u_nk)

# calculate delta G and error
delta_g = np.round(cumulative_data.BAR.f.iloc[-1] * RT, 1)
error = np.round(cumulative_data.BAR.errors.iloc[-1] * RT, 1)

2026-03-18 11:03:26.096 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:634 - Running equilibration detection.


Detecting Equilibrium (includes decorrelating)


2026-03-18 11:03:28.408 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:636 - Start index: 991.
2026-03-18 11:03:28.408 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:637 - Statistical inefficiency: 6.06.
2026-03-18 11:03:28.412 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:643 - Number of uncorrelated samples: 1488.
2026-03-18 11:03:28.419 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:634 - Running equilibration detection.
2026-03-18 11:03:28.528 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:636 - Start index: 491.
2026-03-18 11:03:28.529 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:637 - Statistical inefficiency: 1.35.
2026-03-18 11:03:28.536 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium_detection:643 - Number of uncorrelated samples: 1484.
2026-03-18 11:03:28.546 | DEBUG    | alchemlyb.preprocessing.subsampling:equilibrium

TypeError: Markdown expects text, not np.float64(14.5)